In [1]:
# Import LangSmith and RAG components
from langsmith import Client
from langsmith.evaluation import evaluate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

load_dotenv()

print('Imports ready.')

Imports ready.


Load Clean Vector Store and Build RAG Chain

In [2]:
# Path to clean vector store
chroma_storage_dir = '../../../chroma_db_clean'

# Embeddings and vector store
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(
    persist_directory=chroma_storage_dir,
    embedding_function=embeddings
)

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# LLM and prompt
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer the question using only the provided context. If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}')
])

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# RAG chain
rag_chain = (
    {
        'context': retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print('RAG chain ready.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


RAG chain ready.


Create a Dataset with Reference Answers

In [3]:
# Create LangSmith client
client = Client()

dataset_name = 'rag_eval_reference_dataset'

# Create dataset if it doesn't exist
try:
    dataset = client.create_dataset(
        dataset_name,
        description='RAG evaluation dataset with reference answers'
    )
    print(f'Created dataset: {dataset_name}')

except Exception as e:
    datasets = list(client.list_datasets(dataset_name=dataset_name))
    dataset = datasets[0]
    print(f'Using existing dataset: {dataset_name}')
    
# Define inputs and reference outputs
examples_data = [
    {
        'question': 'What are the common crop diseases and their control methods?',
        'reference': 'Common crop diseases include Cassava Mosaic Disease, Maize Smut, Rice Blast, and Tomato Leaf Curl Virus. Controls include using disease-free cuttings, resistant varieties, crop rotation, and removing infected plants.'
    },
    {
        'question': 'What are the top causes of death in Nigeria?',
        'reference': 'The top causes of death are malaria, lower respiratory infections, HIV/AIDS, diarrheal diseases, road injuries, protein-energy malnutrition, cancer, meningitis, stroke, and tuberculosis.'
    },
    {
        'question': 'What are the symptoms of Cassava Mosaic Disease?',
        'reference': 'Symptoms include yellowing and mottling of leaves, stunted growth, and reduced yield.'
    },
    {
        'question': 'What is Mastitis and how is it managed in dairy animals?',
        'reference': 'Mastitis is an infection of the udder with symptoms like swollen udder, abnormal milk, fever. Management includes good milking hygiene, antibiotics under veterinary guidance, and culling chronic cases.'
    }
]

# Prepare inputs and outputs for dataset
inputs = [{'question': ex['question']} for ex in examples_data]
outputs = [{'answer': ex['reference']} for ex in examples_data]

# Add examples
client.create_examples(
    inputs=inputs,
    outputs=outputs,
    dataset_id=dataset.id
)

print('Examples added to dataset.')

Created dataset: rag_eval_reference_dataset
Examples added to dataset.


Define Target Function
* target function receives a dictionary with 'question' and returns a dictionary with 'answer' (which will be compared to the reference 'answer' in the dataset)

In [4]:
def target_fn(inputs: dict) -> dict:
    question = inputs['question']
    answer = rag_chain.invoke(question)
    return {'answer': answer}

Run Evaluation with "qa" Evaluator


In [11]:
# # Run evaluation directly on the dataset (no need to list examples manually)
# results = evaluate(
#     target_fn,
#     data=dataset_name,          # or dataset.id or dataset object
#     evaluators=["qa"],
#     experiment_prefix="rag_langsmith-qa",
# )

# print("Evaluation completed.")
# print(results)

In [13]:
from langsmith.schemas import Run, Example

# Reuse the same LLM (or a separate judge LLM)
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def qa_correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom evaluator that checks if the generated answer matches the reference answer.
    """
    question = example.inputs["question"]
    prediction = run.outputs["answer"]
    reference = example.outputs.get("answer", "")  # use .get to avoid KeyError

    # Prompt the LLM to judge correctness
    judge_prompt = f"""You are an evaluator for a question-answering system.
Question: {question}
Reference answer: {reference}
System answer: {prediction}
Is the system answer correct and faithful to the reference? Answer 'yes' or 'no' and give a short explanation."""
    
    response = judge_llm.invoke(judge_prompt)
    content = response.content.strip().lower()
    
    score = 1 if "yes" in content else 0
    
    return {
        "key": "correctness",
        "score": score,
        "comment": content
    }

# Run evaluation with the custom evaluator
results = evaluate(
    target_fn,
    data=dataset_name,
    evaluators=[qa_correctness_evaluator],  # pass the function directly
    experiment_prefix="rag_langsmith-qa",
)

print("Evaluation completed.")
print(results)

c:\Users\USER\rag_course\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'rag_langsmith-qa-a2052ec3' at:
https://smith.langchain.com/o/9c84a2e7-b0b5-4ea6-a8a6-38fca7935203/datasets/3f8b1b10-dde3-4de5-826a-eb37e9257022/compare?selectedSessions=90b663d0-2067-455e-b05d-627212a5beb7




0it [00:00, ?it/s]Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
4it [00:12,  3.23s/it]

Evaluation completed.
<ExperimentResults rag_langsmith-qa-a2052ec3>
